            # FahMai Typhoon RAG

            Colab-ready notebook for the FahMai challenge.

            Assumption:
            - You upload this notebook into a workspace that also has the challenge `data/` folder, or you mount a folder where `data/` exists.

            Pipeline:
            1. Install dependencies
            2. Point to the `data/` folder
            3. Build structured chunks from the markdown knowledge base
            4. Run hybrid retrieval (dense + BM25 + title bonus)
            5. Use Typhoon to choose the final answer
            6. Export `submission.csv`
            


In [ ]:
            !pip install -q openai sentence-transformers pythainlp rank-bm25 pandas numpy tqdm
            


In [ ]:
            import os
            from pathlib import Path

            try:
                from google.colab import userdata, files
            except ImportError:
                userdata = None
                files = None

            DATA_DIR = Path("/content/MiniHack3/data")
            if not DATA_DIR.exists():
                DATA_DIR = Path("data")

            TYPHOON_MODEL = "typhoon-v2.5-30b-a3b-instruct"
            EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
            N_QUESTIONS = 100

            api_key = os.getenv("OPENTYPHOON_API_KEY") or os.getenv("TYPHOON_API_KEY")
            if not api_key and userdata is not None:
                for key_name in ("OPENTYPHOON_API_KEY", "TYPHOON_API_KEY"):
                    try:
                        api_key = userdata.get(key_name)
                        if api_key:
                            break
                    except Exception:
                        pass

            if not api_key:
                raise ValueError("Set OPENTYPHOON_API_KEY or TYPHOON_API_KEY before running inference.")

            print("DATA_DIR =", DATA_DIR.resolve())
            print("MODEL =", TYPHOON_MODEL)
            


In [ ]:
from pathlib import Path
MODULE_SOURCE = r'''
import argparse
import csv
import json
import os
import random
import re
import time
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from openai import APIConnectionError, APIStatusError, OpenAI, RateLimitError
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


DEFAULT_TYPHOON_MODEL = "typhoon-v2.5-30b-a3b-instruct"
DEFAULT_EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

SYSTEM_PROMPT = """คุณเป็นระบบตอบคำถามหลายตัวเลือกของร้านฟ้าใหม่
กติกาสำคัญ:
1. ใช้เฉพาะข้อมูลจากบริบทที่ให้มาเท่านั้น
2. เลือกคำตอบได้เพียงหนึ่งข้อ เป็นเลข 1-10
3. ข้อ 9 ใช้เมื่อคำถามเกี่ยวข้องกับร้านฟ้าใหม่ แต่ข้อมูลในบริบทไม่พอจะตอบอย่างมั่นใจ
4. ข้อ 10 ใช้เมื่อคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่ สินค้า แบรนด์ในเครือ นโยบาย บริการ โปรโมชัน หรือข้อมูลร้าน
5. ถ้าบริบทมีหลักฐานตรงตัว เช่น ราคา รุ่น สเปก ระยะเวลา จำนวน หรือเงื่อนไข ให้ยึดหลักฐานตรงตัวนั้น
6. ตอบเป็น JSON บรรทัดเดียวเท่านั้น ในรูปแบบ {"answer": <1-10>, "confidence": <0-1>, "reason": "<สั้นมาก>"}"""

STORE_KEYWORDS = {
    "ฟ้าใหม่",
    "fahmai",
    "สายฟ้า",
    "ดาวเหนือ",
    "คลื่นเสียง",
    "วงโคจร",
    "จุดเชื่อม",
    "zenbyte",
    "novatech",
    "pulsegear",
    "wongkhojon",
    "judchuam",
    "daonuea",
    "saifah",
    "kluensiang",
}

THAI_STOPWORDS = {
    "คือ",
    "และ",
    "กับ",
    "ของ",
    "ที่",
    "ได้",
    "ไหม",
    "มั้ย",
    "หรือ",
    "ครับ",
    "ค่ะ",
    "คะ",
    "หน่อย",
    "หน่อยครับ",
    "หน่อยค่ะ",
    "อะไร",
    "อย่างไร",
    "เท่าไหร่",
    "กี่",
    "รุ่น",
    "สินค้า",
    "ร้าน",
    "ใน",
    "จาก",
    "ให้",
    "มี",
    "เป็น",
    "อยู่",
    "ได้ไหม",
}


def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.replace("฿", " ")
    text = re.sub(r"[,]", "", text)
    text = re.sub(r"[\u200b\xa0]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize_search(text: str) -> List[str]:
    normalized = re.sub(r"[/_\\|:()\-]+", " ", text.lower())
    thai_tokens = word_tokenize(normalized, engine="newmm", keep_whitespace=False)
    extra_tokens = re.findall(r"[a-z0-9\.]+", normalized)
    merged = thai_tokens + extra_tokens
    tokens = []
    for token in merged:
        token = token.strip()
        if not token:
            continue
        if token in THAI_STOPWORDS:
            continue
        if len(token) == 1 and not token.isdigit():
            continue
        tokens.append(token)
    return tokens


def extract_title(text: str, fallback: str) -> str:
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return fallback


def split_markdown_sections(text: str) -> List[Tuple[str, str]]:
    sections: List[Tuple[str, str]] = []
    current_heading = "ภาพรวม"
    buffer: List[str] = []

    for line in text.splitlines():
        if re.match(r"^##+\s+", line):
            content = "\n".join(buffer).strip()
            if content:
                sections.append((current_heading, content))
            current_heading = re.sub(r"^##+\s+", "", line).strip()
            buffer = []
        else:
            buffer.append(line)

    content = "\n".join(buffer).strip()
    if content:
        sections.append((current_heading, content))
    return sections


def split_into_blocks(text: str) -> List[str]:
    raw_blocks = re.split(r"\n\s*\n", text.strip())
    return [block.strip() for block in raw_blocks if block.strip()]


def pack_blocks(blocks: Sequence[str], char_budget: int = 1100) -> List[str]:
    packed: List[str] = []
    current: List[str] = []
    current_len = 0

    for block in blocks:
        block_len = len(block)
        if block_len > char_budget:
            if current:
                packed.append("\n\n".join(current))
                current = []
                current_len = 0
            for start in range(0, block_len, char_budget):
                packed.append(block[start : start + char_budget])
            continue

        if current and current_len + block_len + 2 > char_budget:
            packed.append("\n\n".join(current))
            current = [block]
            current_len = block_len
        else:
            current.append(block)
            current_len += block_len + 2

    if current:
        packed.append("\n\n".join(current))
    return packed


def load_documents(kb_dir: Path) -> List[Dict]:
    documents: List[Dict] = []
    for path in sorted(kb_dir.rglob("*.md")):
        rel_path = path.relative_to(kb_dir).as_posix()
        text = path.read_text(encoding="utf-8")
        documents.append(
            {
                "path": rel_path,
                "text": text,
                "title": extract_title(text, path.stem),
                "category": rel_path.split("/", 1)[0],
            }
        )
    return documents


def build_chunks(documents: Sequence[Dict], char_budget: int = 1100) -> List[Dict]:
    chunks: List[Dict] = []
    for doc_id, doc in enumerate(documents):
        sections = split_markdown_sections(doc["text"])
        for section_heading, section_text in sections:
            blocks = split_into_blocks(section_text)
            for chunk_id, block_group in enumerate(pack_blocks(blocks, char_budget=char_budget)):
                chunk_text = (
                    f"ชื่อเอกสาร: {doc['title']}\n"
                    f"หมวดหมู่: {doc['category']}\n"
                    f"แหล่งข้อมูล: {doc['path']}\n"
                    f"หัวข้อย่อย: {section_heading}\n\n"
                    f"{block_group}"
                )
                title_tokens = set(tokenize_search(doc["title"]))
                path_tokens = set(tokenize_search(doc["path"]))
                chunks.append(
                    {
                        "doc_id": doc_id,
                        "chunk_id": f"{doc_id}-{chunk_id}",
                        "source": doc["path"],
                        "title": doc["title"],
                        "category": doc["category"],
                        "section": section_heading,
                        "text": chunk_text,
                        "title_tokens": title_tokens | path_tokens,
                    }
                )
    return chunks


def load_questions(data_dir: Path) -> List[Dict]:
    rows: List[Dict] = []
    with (data_dir / "questions.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            rows.append(
                {
                    "id": int(row["id"]),
                    "question": row["question"],
                    "choices": {str(i): row[f"choice_{i}"] for i in range(1, 11)},
                }
            )
    return rows


def build_bm25(chunks: Sequence[Dict]) -> Tuple[BM25Okapi, List[List[str]]]:
    tokenized_chunks = [tokenize_search(chunk["text"]) for chunk in chunks]
    return BM25Okapi(tokenized_chunks), tokenized_chunks


def build_embeddings(
    chunks: Sequence[Dict],
    embed_model_name: str = DEFAULT_EMBED_MODEL,
    batch_size: int = 64,
) -> Tuple[SentenceTransformer, np.ndarray]:
    model = SentenceTransformer(embed_model_name)
    texts = [chunk["text"] for chunk in chunks]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return model, np.asarray(embeddings)


def dense_retrieve(
    query: str,
    embed_model: SentenceTransformer,
    chunk_embeddings: np.ndarray,
    top_k: int = 12,
) -> Tuple[np.ndarray, np.ndarray]:
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embeddings, query_embedding.T).reshape(-1)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices, scores[top_indices]


def bm25_retrieve(query: str, bm25: BM25Okapi, top_k: int = 12) -> Tuple[np.ndarray, np.ndarray]:
    tokens = tokenize_search(query)
    scores = np.asarray(bm25.get_scores(tokens))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return top_indices, scores[top_indices]


def question_relevance_bonus(question: str, chunk: Dict) -> float:
    question_tokens = set(tokenize_search(question))
    overlap = len(question_tokens & chunk["title_tokens"])
    bonus = min(overlap, 5) * 0.03

    normalized_question = normalize_text(question)
    normalized_title = normalize_text(chunk["title"])
    if normalized_title and normalized_title in normalized_question:
        bonus += 0.18

    source_slug = normalize_text(Path(chunk["source"]).stem.replace("_", " "))
    if source_slug and source_slug in normalized_question:
        bonus += 0.12

    if chunk["category"] == "policies" and any(
        term in normalized_question
        for term in ["คืน", "ยกเลิก", "จัดส่ง", "ผ่อน", "ชำระ", "รับประกัน", "สมาชิก", "คะแนน"]
    ):
        bonus += 0.04

    if chunk["category"] == "store_info" and any(
        term in normalized_question
        for term in ["สาขา", "ร้าน", "faq", "trade-in", "คริปโต", "crypto", "บริการ"]
    ):
        bonus += 0.04

    return bonus


def hybrid_retrieve(
    question: str,
    chunks: Sequence[Dict],
    embed_model: SentenceTransformer,
    chunk_embeddings: np.ndarray,
    bm25: BM25Okapi,
    final_k: int = 8,
    fetch_k: int = 18,
    rrf_k: int = 60,
) -> List[Dict]:
    dense_idx, _ = dense_retrieve(question, embed_model, chunk_embeddings, top_k=fetch_k)
    bm25_idx, _ = bm25_retrieve(question, bm25, top_k=fetch_k)

    scores: Dict[int, float] = {}
    for rank, idx in enumerate(dense_idx, start=1):
        scores[int(idx)] = scores.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(bm25_idx, start=1):
        scores[int(idx)] = scores.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)

    for idx in list(scores.keys()):
        scores[idx] += question_relevance_bonus(question, chunks[idx])

    top_indices = sorted(scores, key=scores.get, reverse=True)[:final_k]
    return [chunks[idx] for idx in top_indices]


def build_context(chunks: Sequence[Dict]) -> str:
    parts = []
    for rank, chunk in enumerate(chunks, start=1):
        parts.append(
            f"[หลักฐาน {rank}]\n"
            f"แหล่งข้อมูล: {chunk['source']}\n"
            f"ชื่อเอกสาร: {chunk['title']}\n"
            f"หัวข้อย่อย: {chunk['section']}\n"
            f"{chunk['text']}"
        )
    return "\n\n".join(parts)


def exact_choice_match(question: str, choices: Dict[str, str], retrieved_chunks: Sequence[Dict]) -> Optional[int]:
    del question
    context = normalize_text("\n".join(chunk["text"] for chunk in retrieved_chunks))
    support: List[Tuple[int, float]] = []

    for choice_id in range(1, 9):
        choice_text = choices[str(choice_id)]
        normalized_choice = normalize_text(choice_text)
        score = 0.0

        if normalized_choice and normalized_choice in context:
            score += 3.0

        numbers = re.findall(r"\d+(?:\.\d+)?", normalized_choice)
        for number in numbers:
            if number in context:
                score += 0.8

        informative_tokens = [
            token
            for token in tokenize_search(choice_text)
            if token not in STORE_KEYWORDS and len(token) > 1
        ]
        matched = sum(1 for token in informative_tokens if token in context)
        if informative_tokens:
            score += matched / len(informative_tokens)

        support.append((choice_id, score))

    support.sort(key=lambda item: item[1], reverse=True)
    best_choice, best_score = support[0]
    second_score = support[1][1] if len(support) > 1 else 0.0
    if best_score >= 3.3 and best_score - second_score >= 0.75:
        return best_choice
    return None


def parse_json_answer(raw_text: str) -> Optional[Dict]:
    if not raw_text:
        return None

    clean = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
    match = re.search(r"\{.*\}", clean, flags=re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            answer = int(data.get("answer"))
            if 1 <= answer <= 10:
                confidence = data.get("confidence")
                try:
                    confidence = float(confidence) if confidence is not None else None
                except (TypeError, ValueError):
                    confidence = None
                return {
                    "answer": answer,
                    "confidence": confidence,
                    "reason": str(data.get("reason", "")).strip(),
                    "raw": clean,
                }
        except (json.JSONDecodeError, TypeError, ValueError):
            pass

    match = re.search(r"\b([1-9]|10)\b", clean)
    if match:
        answer = int(match.group(1))
        return {"answer": answer, "confidence": None, "reason": "", "raw": clean}
    return None


def build_prompt(question: str, choices: Dict[str, str], retrieved_chunks: Sequence[Dict]) -> str:
    choices_text = "\n".join(f"{key}. {value}" for key, value in choices.items())
    context = build_context(retrieved_chunks)
    return (
        f"บริบทจากฐานข้อมูล:\n{context}\n\n"
        f"คำถาม:\n{question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        "พิจารณาหลักฐานที่มีจริงในบริบทก่อนตอบเสมอ "
        "ถ้าไม่มีหลักฐานพอแต่ยังเป็นเรื่องของฟ้าใหม่ให้ตอบ 9 "
        "ถ้าไม่เกี่ยวข้องกับฟ้าใหม่เลยให้ตอบ 10"
    )


def make_typhoon_client(api_key: Optional[str] = None) -> OpenAI:
    resolved_key = api_key or os.getenv("OPENTYPHOON_API_KEY") or os.getenv("TYPHOON_API_KEY")
    if not resolved_key:
        raise ValueError("Missing Typhoon API key. Set OPENTYPHOON_API_KEY or TYPHOON_API_KEY.")
    return OpenAI(api_key=resolved_key, base_url="https://api.opentyphoon.ai/v1")


def ask_typhoon(
    client: OpenAI,
    messages: Sequence[Dict[str, str]],
    model: str = DEFAULT_TYPHOON_MODEL,
    temperature: float = 0.0,
    max_tokens: int = 180,
    max_retries: int = 6,
) -> str:
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=list(messages),
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except RateLimitError:
            sleep_for = min(2 ** attempt + random.random(), 20)
        except (APIConnectionError, APIStatusError):
            sleep_for = min(2 ** attempt + random.random(), 20)
        time.sleep(sleep_for)
    raise RuntimeError("Typhoon API failed after retries.")


def answer_question(
    client: OpenAI,
    question_row: Dict,
    chunks: Sequence[Dict],
    embed_model: SentenceTransformer,
    chunk_embeddings: np.ndarray,
    bm25: BM25Okapi,
    model_name: str = DEFAULT_TYPHOON_MODEL,
    final_k: int = 8,
    use_second_pass: bool = True,
) -> Dict:
    retrieved_chunks = hybrid_retrieve(
        question_row["question"],
        chunks,
        embed_model,
        chunk_embeddings,
        bm25,
        final_k=final_k,
    )

    heuristic_answer = exact_choice_match(question_row["question"], question_row["choices"], retrieved_chunks)
    raw = ask_typhoon(
        client,
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": build_prompt(question_row["question"], question_row["choices"], retrieved_chunks),
            },
        ],
        model=model_name,
    )
    parsed = parse_json_answer(raw) or {"answer": None, "confidence": None, "reason": "", "raw": raw}

    if heuristic_answer and parsed["answer"] in (None, 9):
        parsed["answer"] = heuristic_answer
        parsed["reason"] = "exact_choice_match"

    confidence = parsed.get("confidence")
    should_retry = (
        use_second_pass
        and parsed["answer"] in (9, 10, None)
        and (confidence is None or confidence < 0.75)
    )

    if should_retry:
        expanded_query = question_row["question"] + "\n" + "\n".join(
            question_row["choices"][str(i)] for i in range(1, 9)
        )
        expanded_chunks = hybrid_retrieve(
            expanded_query,
            chunks,
            embed_model,
            chunk_embeddings,
            bm25,
            final_k=min(final_k + 2, 12),
        )
        raw_retry = ask_typhoon(
            client,
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": build_prompt(question_row["question"], question_row["choices"], expanded_chunks),
                },
            ],
            model=model_name,
        )
        parsed_retry = parse_json_answer(raw_retry)
        if parsed_retry and parsed_retry["answer"] is not None:
            parsed = parsed_retry
            retrieved_chunks = expanded_chunks

    if parsed["answer"] is None:
        parsed["answer"] = heuristic_answer or 9

    return {
        "id": question_row["id"],
        "question": question_row["question"],
        "answer": int(parsed["answer"]),
        "confidence": parsed.get("confidence"),
        "reason": parsed.get("reason", ""),
        "raw_response": parsed.get("raw", raw),
        "sources": [chunk["source"] for chunk in retrieved_chunks],
    }


def run_pipeline(
    data_dir: Path,
    api_key: Optional[str] = None,
    n_questions: int = 100,
    model_name: str = DEFAULT_TYPHOON_MODEL,
    embed_model_name: str = DEFAULT_EMBED_MODEL,
    output_csv: str = "submission.csv",
    diagnostics_csv: Optional[str] = "diagnostics.csv",
) -> pd.DataFrame:
    kb_dir = data_dir / "knowledge_base"
    questions = load_questions(data_dir)
    documents = load_documents(kb_dir)
    chunks = build_chunks(documents)
    bm25, _ = build_bm25(chunks)
    embed_model, chunk_embeddings = build_embeddings(chunks, embed_model_name=embed_model_name)
    client = make_typhoon_client(api_key=api_key)

    results: List[Dict] = []
    for row in tqdm(questions[:n_questions], desc="Answering"):
        result = answer_question(
            client,
            row,
            chunks,
            embed_model,
            chunk_embeddings,
            bm25,
            model_name=model_name,
        )
        results.append(result)
        time.sleep(0.25)

    result_df = pd.DataFrame(results)
    submission = result_df[["id", "answer"]].copy()
    submission.to_csv(output_csv, index=False, encoding="utf-8")

    if diagnostics_csv:
        result_df.to_csv(diagnostics_csv, index=False, encoding="utf-8")

    return result_df


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run the FahMai Typhoon RAG pipeline.")
    parser.add_argument("--data-dir", default="data", help="Path to the challenge data directory.")
    parser.add_argument("--api-key", default=None, help="Typhoon API key. Prefer env vars instead.")
    parser.add_argument("--n-questions", type=int, default=100, help="Number of questions to answer.")
    parser.add_argument("--model-name", default=DEFAULT_TYPHOON_MODEL, help="Typhoon model id.")
    parser.add_argument("--embed-model-name", default=DEFAULT_EMBED_MODEL, help="Embedding model name.")
    parser.add_argument("--output-csv", default="submission.csv", help="Where to write the submission CSV.")
    parser.add_argument(
        "--diagnostics-csv",
        default="diagnostics.csv",
        help="Where to write per-question diagnostics.",
    )
    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    frame = run_pipeline(
        data_dir=Path(args.data_dir),
        api_key=args.api_key,
        n_questions=args.n_questions,
        model_name=args.model_name,
        embed_model_name=args.embed_model_name,
        output_csv=args.output_csv,
        diagnostics_csv=args.diagnostics_csv,
    )
    print(frame[["id", "answer", "confidence"]].head(10).to_string(index=False))
    print(f"\nSaved submission to {args.output_csv}")

'''
Path('fahmai_typhoon_rag.py').write_text(MODULE_SOURCE.strip() + '\n', encoding='utf-8')
print('Wrote local helper module: fahmai_typhoon_rag.py')


In [ ]:
            import pandas as pd
            from fahmai_typhoon_rag import (
                answer_question,
                build_bm25,
                build_chunks,
                build_embeddings,
                hybrid_retrieve,
                load_documents,
                load_questions,
                make_typhoon_client,
            )

            questions = load_questions(DATA_DIR)
            documents = load_documents(DATA_DIR / "knowledge_base")
            chunks = build_chunks(documents)
            bm25, _ = build_bm25(chunks)
            embed_model, chunk_embeddings = build_embeddings(chunks, embed_model_name=EMBED_MODEL)
            client = make_typhoon_client(api_key)

            print("questions:", len(questions))
            print("documents:", len(documents))
            print("chunks:", len(chunks))
            


In [ ]:
            sample = questions[0]
            retrieved = hybrid_retrieve(
                sample["question"],
                chunks,
                embed_model,
                chunk_embeddings,
                bm25,
                final_k=5,
            )

            print("Q1:", sample["question"])
            for idx, chunk in enumerate(retrieved, start=1):
                print(f"\n[{idx}] {chunk['source']} | {chunk['section']}")
                print(chunk["text"][:500])
            


In [ ]:
            demo = answer_question(
                client=client,
                question_row=questions[0],
                chunks=chunks,
                embed_model=embed_model,
                chunk_embeddings=chunk_embeddings,
                bm25=bm25,
                model_name=TYPHOON_MODEL,
            )

            demo
            


In [ ]:
            import time
            from tqdm.auto import tqdm

            results = []
            for row in tqdm(questions[:N_QUESTIONS], desc="Answering"):
                results.append(
                    answer_question(
                        client=client,
                        question_row=row,
                        chunks=chunks,
                        embed_model=embed_model,
                        chunk_embeddings=chunk_embeddings,
                        bm25=bm25,
                        model_name=TYPHOON_MODEL,
                    )
                )
                time.sleep(0.25)

            result_df = pd.DataFrame(results)
            result_df.head()
            


In [ ]:
            submission = result_df[["id", "answer"]].copy()
            submission.to_csv("submission.csv", index=False, encoding="utf-8")
            result_df.to_csv("diagnostics.csv", index=False, encoding="utf-8")

            print(submission.head())
            print("\nSaved submission.csv and diagnostics.csv")

            if files is not None:
                files.download("submission.csv")
            
